<a href="https://colab.research.google.com/github/vishalmmca25-design/build_transfomer_by_skretch/blob/main/trainyourfirstmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import time
import os

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


cpu


import data set

In [ ]:
import os
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(DATA_URL, "input.txt")

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Length of dataset in characters:", len(text))
print(text[:200])

Length of dataset in characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


charter level tokenzation


In [ ]:
# Build character vocabulary
chars = sorted(set(text))
vocab_size = len(chars)

# Character -> integer
char_to_idx = {c: i for i, c in enumerate(chars)}

# Integer -> character
idx_to_char = {i: c for i, c in enumerate(chars)}

# Encode
def encode(s):
    return [char_to_idx[c] for c in s]

# Decode
def decode(indices):
    return ''.join(idx_to_char[i] for i in indices)


train and testing split

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

print(train_data.shape)

torch.Size([1003854])


In [ ]:
#betching grab rendom chunks of test
#why we o that
# input:'they enj'
#target:'hey enjo'
def get_batch(split,batch_size,block_size):
    data = train_data if split == 'train' else test_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x= torch.stack([data[i:i+block_size] for i in ix])
    y= torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

rms norm

In [ ]:
class RMSNorm(nn.Module):
  def __init__(self, dim, eps=1e-8):
    super().__init__()
    self.scale = dim ** -0.5
    self.eps = eps
    self.g = nn.Parameter(torch.ones(dim))

  def forward(self,x):
    res=torch.sqrt(x.pow(2).mean(dim=-1,keepdim=True)+self.eps)
    return (x/res)*self.weight

#prevent from overfiting
##dropout rendomly zeroes
#not rely on single  feture
  DROPOUT=0.2





ROPE rotary position embedding

In [ ]:
import torch

def precompute_rope_freqs(head_dim, max_seq_len, base=10000.0):
    # head_dim must be even
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))

    positions = torch.arange(max_seq_len).float()

    angles = torch.outer(positions, freqs)

    return torch.cos(angles), torch.sin(angles)


def apply_rope(x, cos, sin):
    # x shape: (batch, heads, seq_len, head_dim)

    seq_len = x.shape[2]

    cos = cos[:seq_len].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len].unsqueeze(0).unsqueeze(0)

    x1 = x[..., ::2]
    x2 = x[..., 1::2]

    out1 = x1 * cos - x2 * sin
    out2 = x1 * sin + x2 * cos

    return torch.stack((out1, out2), dim=-1).flatten(-2)



Grouped query attention

In [ ]:
def forward(self, x, rope_cos, rope_sin):

    b, seq_len, _ = x.shape

    # projections
    q = self.q_proj(x)
    k = self.k_proj(x)
    v = self.v_proj(x)

    # reshape → (b, heads, seq, head_dim)
    q = q.view(b, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
    k = k.view(b, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
    v = v.view(b, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

    # RoPE
    q = apply_rope(q, rope_cos, rope_sin)
    k = apply_rope(k, rope_cos, rope_sin)

    # repeat KV heads for GQA
    k = repeat_kv(k, self.n_rep)
    v = repeat_kv(v, self.n_rep)

    # scaled dot-product attention
    scale = 1.0 / math.sqrt(self.head_dim)
    scores = torch.matmul(q, k.transpose(-2, -1)) * scale   # (b, h, seq, seq)

    # causal mask (lower triangular)
    mask = torch.tril(
        torch.ones(seq_len, seq_len, device=x.device)
    ).bool()

    scores = scores.masked_fill(~mask, float("-inf"))

    # softmax
    attn = F.softmax(scores, dim=-1)

    # dropout (if defined)
    attn = self.dropout(attn)

    # weighted sum
    out = torch.matmul(attn, v)

    # merge heads
    out = (
        out.transpose(1, 2)
           .contiguous()
           .view(b, seq_len, self.d_model)
    )

    return self.o_proj(out)



